Chapter 1 of Victor Dibia's [Designing Multi-Agent Systems](https://www.manning.com/books/designing-multi-agent-systems) sets up the vocabulary the rest of the book builds on: what an agent is, which task properties push past a single agent, and the two families of orchestration. I'm reading it as groundwork for the orchestration module of my agent study plan, where the open question is whether a multi-agent setup fixes the top failure class of a single-agent loop.

The chapter closes with a first working system, a haiku writer and a critic taking turns under a round-robin orchestrator. These notes record the ideas I want to keep and how they connect to my own experiments.


## When one model is not enough

The chapter grades tasks on three levels. A factual question is model-level: training data is enough. A question about a stock price that moved after training is agent-level: the model will guess unless something plans the steps (fetch live data, then answer) and executes them with tools. "Build an app that shows stock prices, buys stock, and files taxes" is multi-agent level: it needs several kinds of expertise, and the path to a working app only emerges from acting and reacting to results.

The useful part of this ladder is that each rung names the capability that is missing at the rung below: retrieval, then planning plus tools, then specialization plus adaptation.


## What is an agent?

The book defines an agent by four capabilities: reason (apply logic to context, including noticing that "today" needs live data), act (call APIs, run code, search), communicate (with users, other agents, and external systems), and adapt (retry, switch data sources, adjust the query after an error).

Three components enable them:

- **Model** — the reasoning engine, usually an LLM, but the chapter allows custom logic or just-in-time human input to drive decisions too.
- **Tools** — general-purpose ones like a code executor or UI driver versus domain-specific ones like a particular API call.
- **Memory** — short-term working context for the current task versus long-term knowledge that persists; also split by who curates it, the application or the agent itself.

Agents run an action-perception loop: act, observe, adapt, repeat. The chapter calls the implementation style *conversational programming*: every step of the loop is a message, which is why chat-trained models fit it and how message-passing frameworks like AutoGen work.


## Why some tasks need more than one agent

Four task characteristics motivate multiple agents, and it is usually their combination that does:

1. **Planning** — the task decomposes into steps that must each succeed.
2. **Diverse expertise** — steps want different specialists (a UX agent, an Android agent), which gives separation of concerns a single generic instruction set struggles to achieve.
3. **Extensive context** — the full history overwhelms one context window. LLMs attend to the start and end of long inputs and neglect the middle, and the chapter borrows cognitive load theory to make the same point. The multi-agent answer is *selective context provisioning*: each agent sees only the slice of history relevant to its step.
4. **Adaptive solutions** — the solution is unknown until actions are taken; data sources fail, APIs change. Writing a deterministic pipeline stops being possible, so agents self-orchestrate instead.

The chapter cites debate-style multi-agent setups improving factuality, and role-separated generator/adjudicator pairs improving output diversity, as research support.

For my study plan this connects directly to the orchestration module: these four characteristics are the checklist for deciding whether a multi-agent fix addresses the top failure class, and the comparison has to happen at matched token budget.


## Two families of orchestration

Orchestration is the book's central term: the mechanisms that decide how agents share information and who controls the order of execution. Two families:

- **Multi-agent workflows** — control flow is programmed up front. Predictable and repeatable; the book's example is a document pipeline with extraction, analysis, and formatting stages in a fixed sequence.
- **Autonomous orchestration** — an AI model makes routing decisions at runtime, so control flow emerges from agent interactions. Better for tasks where the strategy cannot be predetermined.

The same agent anatomy (model, tools, memory) sits under both; the difference is only who decides the next step. A side note I want to keep: distributed agents across organizational boundaries are deferred to a later chapter on protocols, with the argument that going distributed early is usually overengineering, and that the vast majority of use cases are served by a single process.


## Choosing an architecture

The decision framework walks a ladder of questions:

- No actions beyond text generation? Call the model directly.
- Actions needed, and the approach is a known workflow? Use a single agent with tools, unless the steps want distinct expertise, in which case a multi-agent workflow fits.
- Approach unknown, solution emerging through exploration? That is where autonomous multi-agent systems earn their place.

The rule of thumb is to choose the simplest architecture that covers the requirements, because each level adds cost, latency, and new failure modes. The chapter points to its own evaluation results as evidence: on simple reasoning tasks, direct model calls scored best at a fraction of the token cost, while on research tasks needing web search, plain models collapsed and tool-using agents did well. Architecture is a cost-versus-capability trade, not a sophistication contest.

This is closer to the "don't build multi-agents by default" argument than I expected from a multi-agent book, which makes the four task characteristics above the more important: they are the cases where the extra machinery pays.


## First system: a poet and a critic

The chapter's first runnable system is two agents: a haiku poet and a critic instructed to give a few specific suggestions and reply with a marker word when satisfied. A `RoundRobinOrchestrator` runs them in a shared turn-taking loop, and termination combines a message cap with a text trigger, so the run ends when either the critic approves or the budget runs out.

The agents come from `picoagents`, the small library the book builds from scratch starting in chapter 4. This chapter only previews its surface: an `Agent` with name, description, instructions, and a model client; an orchestrator over a list of agents; and composable termination conditions. The book also shows a one-line web UI for watching the conversation, built later in chapter 8.

The example below is my adaptation of that system for a different subject and task, kept here as the shape of the API to keep in mind while reading chapter 4.


In [ ]:
# Adapted from chapter 1's poet-critic example: a different subject and task,
# same round-robin shape. Needs `pip install picoagents` and OPENAI_API_KEY.
import asyncio
import os

from picoagents import Agent
from picoagents.llm import OpenAIChatCompletionClient
from picoagents.orchestration import RoundRobinOrchestrator
from picoagents.termination import MaxMessageTermination, TextMentionTermination

client = OpenAIChatCompletionClient(
    model="gpt-4.1-mini", api_key=os.getenv("OPENAI_API_KEY")
)

poet = Agent(
    name="poet",
    description="Writes haiku.",
    instructions="You are a haiku poet. Write one haiku per turn.",
    model_client=client,
)

critic = Agent(
    name="critic",
    description="Reviews haiku.",
    instructions=(
        "You review haiku. Give 2-3 specific, actionable suggestions, "
        "or reply with APPROVED alone when no changes are needed."
    ),
    model_client=client,
)

termination = MaxMessageTermination(max_messages=8) | TextMentionTermination(
    text="APPROVED"
)
orchestrator = RoundRobinOrchestrator(
    agents=[poet, critic], termination=termination, max_iterations=4
)


async def main() -> None:
    async for message in orchestrator.run_stream("Write a haiku about autumn rain"):
        print(message)


asyncio.run(main())


## Takeaways

- The vocabulary to keep: an agent reasons, acts, communicates, and adapts over model, tools, and memory; orchestration is control flow plus information sharing; workflows program the control flow, autonomous systems let the model decide it at runtime.
- The four task characteristics (planning, diverse expertise, extensive context, adaptive solutions) are a practical checklist for my orchestration module: multi-agent is worth testing when they cluster, and the comparison must be at matched token budget.
- The architecture ladder ends in "simplest thing that works," backed by the book's own evaluation numbers. That aligns this book with the skeptical multi-agent reading I already had queued.
- Next: chapter 2, the pattern taxonomy, covering sequential, conditional, and parallel workflows, and plan-based, handoff, and group-chat autonomous patterns.
